# Esercizio 1 — Object Detection YOLO (demo)

Dimostrazione del detector **YOLO** su un subset di **Object365** (8 classi).

Tutta la logica di detection è **implementata a mano**: griglia di celle, anchor box, costruzione dei target, **loss YOLO**, **IoU** e **non-max suppression**. Il solo estrattore di feature è una **ResNet18 pre-addestrata su ImageNet**, poi fine-tunata (*transfer learning*, argomento del corso). Impostando `PRETRAINED_BACKBONE = False` in `config.py` si usa invece il backbone convoluzionale scritto interamente da zero.

Il notebook mostra le predizioni con bounding box, la valutazione **precision/recall a IoU≥0.5** e la detection su una foto a scelta.

Richiede un modello già addestrato (`outputs/best.pt`), prodotto con `python src/train.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import torch
from config import DEVICE, CLASS_NAMES, OUTPUT_DIR, PRETRAINED_BACKBONE, GRID, load_anchors
from model import YOLO
from detect import detect, draw

print('Device:', DEVICE)
print('Classi:', CLASS_NAMES)
print('Griglia:', f'{GRID}x{GRID}')
print('Backbone:', 'ResNet18 pre-addestrata (transfer learning)' if PRETRAINED_BACKBONE
      else 'convoluzionale scritto da zero')

## 1. Carico il modello addestrato

In [ ]:
anchors = load_anchors()
model = YOLO().to(DEVICE)
ckpt = torch.load(Path(OUTPUT_DIR) / 'best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print('Modello caricato (epoch', ckpt['epoch'] + 1, ', val loss', round(ckpt.get('best_val', float('nan')), 3), ')')

## 2. Detection su alcune immagini di validazione

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from dataset import Object365Detection

val = Object365Detection('val')
sample_ids = val.ids[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, name in zip(axes.ravel(), sample_ids):
    img = Image.open(val.img_dir / f'{name}.jpg').convert('RGB')
    results = detect(model, img, anchors, DEVICE, conf_thresh=0.25, nms_thresh=0.45)
    ax.imshow(draw(img, results))
    ax.set_title(f'{len(results)} oggetti')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Valutazione quantitativa — precision/recall a IoU≥0.5

In [ ]:
from evaluate import evaluate
precision, recall = evaluate(str(Path(OUTPUT_DIR) / 'best.pt'), conf_thresh=0.25)

## 4. Detection su una tua foto

Imposta `MY_PHOTO` sul percorso di una foto (es. scattata con la fotocamera) contenente uno degli oggetti tra: Person, Car, Chair, Bottle, Cup, Lamp, Hat, Book.

In [ ]:
MY_PHOTO = None  # es. '/percorso/foto.jpg'

if MY_PHOTO:
    img = Image.open(MY_PHOTO).convert('RGB')
    results = detect(model, img, anchors, DEVICE, conf_thresh=0.25, nms_thresh=0.45)
    for cls, score, box in results:
        print(f'{CLASS_NAMES[cls]:8s} {score:.2f}')
    plt.figure(figsize=(9, 9)); plt.imshow(draw(img, results)); plt.axis('off'); plt.show()
else:
    print('Imposta MY_PHOTO con il percorso di una foto.')